In [2]:
import pandas as pd

df= pd.read_csv('advanced_pool4_10M.csv')


new_df=df[(df['NumAttractors']<=64) & (df['MACA_or_not']==1)]

new_df.head()

,RuleVectorID,RuleVector,NumAttractors,NumFixedPointAttractors,NumCyclicAttractors,AvgCycleLength,MaxCycleLength,MaxChainLength,PointAttractorDensity,Reversible,ImpurityCells,ImpurityPercent,basins,MACA_or_not
59,HybridRV_800005,79-204-204-218-204-204-204-94,60,60,0,1.0,1,1,23.43750,0,3,37.5,191:190;215:214-87-86;213:212-85-84;217:216-20...,1
61,HybridRV_200006,223-204-223-204-204-213-93-204,35,35,0,1.0,1,2,13.67190,0,4,50.0,223:222;219:218;221:220-217-216;215:214-211-21...,1
100,HybridRV_900009,204-239-88-217-214-204-148-157,9,9,0,1.0,1,5,3.51562,0,6,75.0,187:253-247-189-183-243-125-237-119-167-115-22...,1
115,HybridRV_700011,79-64-138-168-204-254-204-13,14,14,0,1.0,1,2,5.46875,0,6,75.0,123:251-219-91;177:176-145-144-50-49-178-48-18...,1
151,HybridRV_200015,223-204-204-224-204-8-204-204,52,52,0,1.0,1,1,20.31250,0,3,37.5,231:230;225:233-232-224;223:255-247-246-222-25...,1


In [5]:
import numpy as np
from itertools import combinations


cell_count = 8

def hamming_distance(bin1, bin2):
    return sum(c1 != c2 for c1, c2 in zip(bin1, bin2))

# We will collect per-row summary statistics
results = []

for idx, row in new_df.iterrows():
    basin_str = row["basins"]
    if basin_str == "0" or pd.isna(basin_str):
        continue

    min_hamming = None
    max_hamming = None
    all_distances = []

    # Split attractors
    attractor_parts = basin_str.split(";")
    for part in attractor_parts:
        if ":" not in part:
            continue
        attractor, preds = part.split(":")
        state_nums = preds.split("-")
        if len(state_nums) < 2:
            continue  # No pairs

        # Convert to binary strings
        bin_states = [
            format(int(s.strip()), f"0{cell_count}b")
            for s in state_nums
            if s.strip() != ""
        ]

        # Compute all combinations
        for a, b in combinations(bin_states, 2):
            d = hamming_distance(a, b)
            all_distances.append(d)

    if all_distances:
        min_hamming = min(all_distances)
        max_hamming = max(all_distances)
        avg_hamming = np.mean(all_distances)
    else:
        min_hamming = max_hamming = avg_hamming = np.nan

    results.append({
        "RuleVectorID": row["RuleVectorID"],
        "Attractors": row["NumAttractors"],
        "Max_Chain_Length": row["MaxChainLength"],
        "MinIntraBasinHamming": min_hamming,
        "MaxIntraBasinHamming": max_hamming,
        "AvgIntraBasinHamming": avg_hamming,
        "NumDistances": len(all_distances)
    })

hamming_df = pd.DataFrame(results)

display(hamming_df.head())  

,RuleVectorID,Attractors,Max_Chain_Length,MinIntraBasinHamming,MaxIntraBasinHamming,AvgIntraBasinHamming,NumDistances
0,HybridRV_800005,60,1,1,3,1.609195,348
1,HybridRV_200006,35,2,1,4,2.040469,939
2,HybridRV_900009,9,5,1,6,2.938266,4649
3,HybridRV_700011,14,2,1,6,2.860454,4142
4,HybridRV_200015,52,1,1,3,1.675214,468
